# 02 · Warm-start diagnostics (Chapter 7, Section 7.3)

| Output | Thesis | Source (`Methodolody_claims_verifications_for_chapter_5`) |
|---|---|---|
| control table: baseline / rarity hint / full model | Table 7.10 | cell 6 |
| META-DDIE frequency groups | Table 7.9 | cell 33, part 2 |
| rare nodes with / without a same-class neighbour | Table 7.11 | cell 14 |
| kNN label purity of the retrieved neighbours | Ch5 claim (24.5% vs 10.7%) | cell 5 |

Needs the baseline and concat checkpoints from notebook 01 (or the thesis checkpoints on the full graph).
Left out on purpose: the concat-vs-set fusion ablation, whose saved predictions came from a split overlapping training.

**Open issue:** the group sizes printed in Table 7.9 of the thesis (216,415 / 176) do not match what this code produces on the thesis checkpoints (166,570 / 39 in the original run). Check the output below before reusing that table.

In [1]:
ON_COLAB  = False
REPO_DIR  = "/home/ppcrniai/Documents/ddi-linegraph"
DATA_ROOT = "/home/ppcrniai/Documents/ddi-linegraph/ddi_data"
WORK_ROOT = "/home/ppcrniai/Documents/ddi-linegraph/ddi_runs"
TEST_MODE = True
N_TEST_DRUGS = 800   # must match the test graph built in notebook 00

In [2]:
import os, sys
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists(os.path.join(REPO_DIR, "pyproject.toml")):
        raise SystemExit(f"Repo not found at {REPO_DIR}. Upload or clone it there first.")
    !bash {REPO_DIR}/setup/colab_setup.sh
# the editable install is only picked up by a fresh interpreter; make it importable now
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

In [3]:
import json, gc
import numpy as np
import torch
import dgl
import warnings
warnings.filterwarnings("ignore", message="Dataloader CPU affinity opt is not enabled")

from ddi.paths import get_paths
paths = get_paths(DATA_ROOT, WORK_ROOT, local_bin_dir=("/content" if ON_COLAB else None))
paths.make_output_dirs()
paths.show()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| TEST_MODE:", TEST_MODE)

  data_root              /home/ppcrniai/Documents/ddi-linegraph/ddi_data
  work_root              /home/ppcrniai/Documents/ddi-linegraph/ddi_runs
  raw_data_dir           /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data   (does not exist yet)
  drug_db                /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data/Drug_META_DDIE.db   (does not exist yet)
  event_db               /home/ppcrniai/Documents/ddi-linegraph/ddi_data/data/event_META_DDIE_5foldLabel.db   (does not exist yet)
  graph_zip              /home/ppcrniai/Documents/ddi-linegraph/ddi_data/ddi_graph_and_linegraph.zip   (does not exist yet)
  graph_bin              /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/ddi_graph_and_linegraph.bin   (does not exist yet)
  thesis_warm_ckpt_dir   /home/ppcrniai/Documents/ddi-linegraph/ddi_data/thesis_smart_fusion_whole_graph_checkpoints   (does not exist yet)
  thesis_hidden384_dir   /home/ppcrniai/Documents/ddi-linegraph/ddi_data/thesis_hidden384_experiment   (does not ex

/home/ppcrniai/Documents/ddi-linegraph/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from ddi import graph as G
from ddi.splits import filter_and_split_graph, save_split, check_split

if TEST_MODE:
    pair = os.path.join(paths.test_graph_dir, f"test_pair_{N_TEST_DRUGS}.bin")
    assert os.path.exists(pair), "Run notebook 00 first to build the test subgraph."
    subgraph, line_subgraph = G.load_graph_pair(pair)
else:
    G.ensure_graph_bin(paths.graph_zip, paths.graph_bin)
    subgraph, line_subgraph = G.load_graph_pair(paths.graph_bin)   # 640M edges: high-RAM only
print(subgraph); print(line_subgraph)

line_subgraph = G.attach_endpoint_features(subgraph, line_subgraph)
g_lg_filt = filter_and_split_graph(line_subgraph, label_key="label")
del line_subgraph; gc.collect()

# write the split once, then check every later run against it
RUN_TAG = f"test{N_TEST_DRUGS}" if TEST_MODE else "full"
split_path = os.path.join(paths.warm_dir, f"split_{RUN_TAG}.pt")
if os.path.exists(split_path):
    check_split(g_lg_filt, split_path)
else:
    save_split(g_lg_filt, split_path)
NUM_CLASSES = int(g_lg_filt.ndata["label"].max().item() + 1)
print("NUM_CLASSES:", NUM_CLASSES)

Graph(num_nodes=800, num_edges=81242,
      ndata_schemes={'orig_drug_id': Scheme(shape=(), dtype=torch.int64), 'id': Scheme(shape=(), dtype=torch.int64), 'smiles_embedding': Scheme(shape=(384,), dtype=torch.float32)}
      edata_schemes={'orig_edge_id': Scheme(shape=(), dtype=torch.int64), 'label': Scheme(shape=(), dtype=torch.int64), 'id': Scheme(shape=(), dtype=torch.int64), 'fused_embedding': Scheme(shape=(384,), dtype=torch.float32)})
Graph(num_nodes=81242, num_edges=8520237,
      ndata_schemes={}
      edata_schemes={})
Filtered graph: 81223 nodes | Classes kept: 123 (123 actually present)
Dropping 7 class(es) too rare in val/test split: [97, 120, 132, 137, 145, 164, 168]
Split summary: Train=56856 | Val=12180 | Test=12180
split matches /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/split_test800.pt
NUM_CLASSES: 181


In [5]:
from ddi import diagnostics as D

USE_THESIS_CHECKPOINTS = False
if USE_THESIS_CHECKPOINTS:
    assert not TEST_MODE
    ck = paths.thesis_warm_ckpt_dir
    knn_cache = paths.thesis_knn_cache_k5
else:
    ck = os.path.join(paths.warm_dir, f"checkpoints_{RUN_TAG}")
    knn_cache = os.path.join(paths.warm_dir, f"knn_{RUN_TAG}", "knn_cache_k5.pt")

npz = os.path.join(paths.diag_dir, f"control_preds_{RUN_TAG}{'_thesis' if USE_THESIS_CHECKPOINTS else ''}.npz")
if os.path.exists(npz):
    ctrl = dict(np.load(npz))
else:
    ctrl = D.control_logits(g_lg_filt, os.path.join(ck, "ckpt_baseline.pt"),
                            os.path.join(ck, "ckpt_proto_concat_rare_only_k5.pt"),
                            device, knn_cache_path=knn_cache)
    np.savez(npz, **ctrl)
train_counts, tail = D.train_counts_and_tail(g_lg_filt, NUM_CLASSES, thr=15)
print(f"rare classes: {int(tail.sum())} | rare test interactions: {int(tail[ctrl['y_true']].sum())}")

Loaded kNN cache from /home/ppcrniai/Documents/ddi-linegraph/ddi_runs/warmstart/knn_test800/knn_cache_k5.pt: torch.Size([81223, 5])
rare classes: 96 | rare test interactions: 50


## Table 7.10: rarity-signal control

In [6]:
t710 = D.control_table(ctrl, tail, NUM_CLASSES)

model                        tailF1  macroEvt  macro212   rare correct
baseline                     0.0000    0.0656    0.0685         0/50
+ rarity hint                0.0501    0.0924    0.0951        12/50
+ prototype smoothing        0.0701    0.0860    0.0886        14/50


## Table 7.9: META-DDIE frequency groups

In [7]:
t79 = D.frequency_groups(ctrl, NUM_CLASSES)


baseline
  group                   classes   test n   macroF1
  common (#1-#175)            175     9341    0.0675
  fewer (#176-#204)             5        1    0.0000
  rare (#205-#227)              0        0    0.0000

full model
  group                   classes   test n   macroF1
  common (#1-#175)            175     9341    0.0885
  fewer (#176-#204)             5        1    0.0000
  rare (#205-#227)              0        0    0.0000


## Table 7.11: same-class neighbour split

In [8]:
cache = torch.load(knn_cache, map_location="cpu")
t711 = D.label_dependence(ctrl, g_lg_filt, cache["knn_idx"], cache["knn_sim"], train_counts)


RARE THRESHOLD < 15   (96 classes, 50 test nodes)
group        n  base ok  aug ok   gain  W->R  R->W   rank_b   rank_a  d rank   mass  n_train
A same      27        0       9      9     9     0     52.9      6.4    46.6  0.350      7.9
B none      23        0       5      5     5     0     66.6      4.7    61.9  0.000      7.7
all         50        0      14     14    14     0     59.2      5.6    53.6  0.189      7.8

augmented-model accuracy, A vs B: Fisher exact p = 0.529
rank improvement A +46.6 | B +61.9 | diff 95% CI [-30.6, +0.1] | Mann-Whitney p = 0.078

RARE THRESHOLD < 20   (105 classes, 82 test nodes)
group        n  base ok  aug ok   gain  W->R  R->W   rank_b   rank_a  d rank   mass  n_train
A same      38        0       9      9     9     0     42.8     13.3    29.6  0.397     10.3
B none      44        0       5      5     5     0     50.6     21.2    29.3  0.000     11.7
all         82        0      14     14    14     0     47.0     17.5    29.4  0.184     11.0

augmen

## kNN label purity

In [9]:
purity = D.knn_purity(g_lg_filt, cache["knn_idx"], cache["knn_sim"])
json.dump({"table_7_10": t710, "table_7_9": t79, "table_7_11": {str(k): v for k, v in t711.items()},
           "knn_purity": purity},
          open(os.path.join(paths.diag_dir, f"diagnostics_{RUN_TAG}.json"), "w"), indent=2)


RARE nodes (class train count < 20)  (n = 548)
  neighbours sharing the node's class      : 13.9%
  prototype MASS on same-class neighbours  : 17.0%
  nodes with >=1 same-class neighbour      : 43.2%
  nodes whose nearest neighbour is same-cls: 13.5%
  distinct classes among the neighbours    : 2.97

RARE nodes (class train count < 15)  (n = 342)
  neighbours sharing the node's class      : 12.3%
  prototype MASS on same-class neighbours  : 16.1%
  nodes with >=1 same-class neighbour      : 43.9%
  nodes whose nearest neighbour is same-cls: 13.2%
  distinct classes among the neighbours    : 2.95

COMMON nodes (not smoothed)  (n = 80675)
  neighbours sharing the node's class      : 35.9%
  prototype MASS on same-class neighbours  : 37.7%
  nodes with >=1 same-class neighbour      : 74.5%
  nodes whose nearest neighbour is same-cls: 35.7%
  distinct classes among the neighbours    : 2.73

chance rate (two random training interactions share a class): 10.83%
